# Building an Image Dataset by Scraping

Every image dataset you have trained on so far arrived clean. This one will not. Here you pick
your own categories, search the web for them, and download whatever comes back — which is how
most real image datasets begin, and why most of the work in an image project happens before any
model is trained.

This notebook is your starting point for P2 Notebook 1. It demonstrates the provided scraper on
a small example. Your job is to replace the example with your own classes, scrape at full scale,
and add the accounting and manifest that the assignment asks for.

## Learning objectives

- Choose image classes that make a learnable, non-trivial classification problem
- Write several search queries per class and explain why one query is not enough
- Run a scraper with arguments you can justify, rather than defaults you inherited
- Account for the gap between images requested, images downloaded, and images readable
- Export a manifest that records what you actually trained on

## Background

You need to be comfortable with paths and folders (`pathlib`), with opening images in PIL, and
with building a DataFrame from a list of dictionaries. The scraping mechanics themselves are
handled for you by `ddgs_scraper.py`, which sits next to this notebook — you are expected to read
that module before you run it, because this assignment asks you to justify the arguments you pass.

Scraped images are copyrighted by whoever made them. They are fine to use for coursework, but do
**not** redistribute them, do **not** commit them to your repo beyond the one sample per class the
prompt allows, and do **not** publish your dataset.

## This notebook covers

1. What the scraper does
2. Declaring your classes and queries
3. Running the scraper
4. Accounting for what came back
5. A first look at the images
6. Turning the records into a manifest

**Prerequisites:** none — this is the first notebook of P2.

**Dataset:** yours. Built here from DuckDuckGo image search.

**References:** <https://pypi.org/project/ddgs/>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# The scraper lives in ddgs_scraper.py, in this same folder. Keep the two files
# together — if you move this notebook into your repo, move the module with it.
import ddgs_scraper

## 1. What the scraper does

`ddgs_scraper.py` wraps the `ddgs` package, which queries DuckDuckGo image search. The pieces you
will actually call are:

| Function | What it does |
|---|---|
| `search_images(query, ...)` | One search. Returns raw result records — image URL, page URL, title, reported size |
| `download_image(url, ...)` | Fetches one URL and returns an RGB PIL image, or `None` if it isn't one |
| `passes_image_filters(img, ...)` | Checks a downloaded image against size and aspect-ratio limits |
| `scrape_class(label, queries, ...)` | All queries for one class, into one folder |
| `build_image_dataset(class_queries, ...)` | Every class. **This is the one you call.** |

Two things the scraper deliberately does *not* do, because they are your job in Notebook 2:
it does not check for duplicate images, and it does not judge whether the picture is actually of
the thing you searched for.

Read the module before running it. The arguments below are not magic numbers and you will be asked
to defend them.

In [ ]:
# The filters every downloaded image has to pass. Both are measured on the image
# that actually arrived, not on the size the search engine claimed.
help(ddgs_scraper.build_image_dataset)

## 2. Declaring your classes and queries

### 2.1 Choosing classes

Your categories are your choice, within the constraints in the assignment prompt: 3–10 classes, at
least 200 usable images each *after* cleaning, not a standard benchmark, and visually
distinguishable but not trivially so.

The last one is the constraint people get wrong. Classes a model separates at 99% teach you
nothing, and classes that are genuinely ambiguous teach you nothing either. Aim for categories a
person could sort reliably but not instantly — different hand tools, dog breeds, architectural
styles, cloud types, guitar body shapes.

### 2.2 Why several queries per class

A single search returns a narrow, repetitive slice of a category. Search "hammer" and you get a
thousand near-identical product photos of claw hammers on white backgrounds. Your model will learn
"white background" and fall over on the first photo taken in a garage.

Several queries per class widen what the class looks like, and they are also the main reason a
class survives deduplication with enough images left. Put your best query first — the scraper works
through them in order and stops once the class is full.

In [ ]:
# ---------------------------------------------------------------
# REPLACE THIS with your own classes. It is here to show the shape.
# ---------------------------------------------------------------
CLASS_QUERIES = {
    'acoustic guitar': [
        'acoustic guitar',
        'dreadnought acoustic guitar',
        'classical nylon string guitar',
    ],
    'electric guitar': [
        'electric guitar',
        'stratocaster electric guitar',
        'les paul electric guitar',
    ],
}

for label, queries in CLASS_QUERIES.items():
    print(f'{label:<20} {len(queries)} queries')

## 3. Running the scraper

The settings below are small on purpose — this is a demonstration run that should finish in a
minute or two. **For your actual dataset you will need much larger numbers.**

A rough guide: to end up with 200 usable images per class after Notebook 2's cleaning, plan to
save 300–400 per class here. You will lose images to wrong subjects, duplicates, and files that
turn out not to open.

Each argument is a decision you have to justify in your writeup:

- `images_per_class` — how many you keep. Set it above your target, not at it
- `results_per_query` — how many search results to request. Most will not survive download and filtering
- `min_resolution` — a floor on quality. Small images look terrible resized up to 224×224
- `aspect_ratio_range` — rejects banners and infographics, which are rarely pictures of your subject
- `safesearch` — leave it on unless you have a specific reason
- `region` — changes what you get back. Say so if you change it

### 3.1 Expect to be rate limited

DuckDuckGo throttles hard, and when it does it reports **"No results found"** rather than an
explicit rate-limit error. That looks identical to a query that genuinely matched nothing.

The scraper retries a few times with backoff before believing it, and marks any class that ends
with zero images as `empty` in the summary rather than `ok`. You still need to watch for it:

- A class reported as `empty`, or with far fewer images than its neighbours, is almost always
  throttling — not a bad query
- The fix is to wait several minutes and re-run **just that class**. Images already saved stay put,
  but note that filenames restart at `_0000`, so scrape a class into a clean folder or renumber
- Scraping five classes at full scale in one uninterrupted run is unlikely to work. Plan to do it
  in batches over more than one sitting
- This is the single most common reason a class comes up short of 200. Start early

In [ ]:
records, summary = ddgs_scraper.build_image_dataset(
    class_queries=CLASS_QUERIES,
    save_dir='images',
    images_per_class=20,          # demo scale — raise this for your real dataset
    results_per_query=60,
    aspect_ratio_range=(0.5, 2.0),
    min_resolution=(200, 200),
    per_image_timeout=10,
    safesearch='moderate',
    region='us-en',
)

## 4. Accounting for what came back

The assignment asks for a table tracking each class from **requested** through **downloaded** to
**readable**. The scraper gives you the first two. The third is yours to produce, and it is the
step that catches the files that downloaded fine but will not open.

A count you cannot explain is the problem the prompt is warning about — not the fact that images
were lost.

In [ ]:
counts = pd.DataFrame([
    {
        'class': label,
        'queries': info['queries'],
        'requested': info['queries'] * 60,   # results_per_query above
        'saved': info['saved'],
        'status': info['status'],
    }
    for label, info in summary['classes'].items()
])

print(f"Total saved: {summary['total_saved']}")
print(f"Output folder: {summary['save_dir']}")
counts

### 4.1 What you still have to add

The scraper saved only images that downloaded and passed the filters, so everything on disk right
now is at least readable. That will **not** stay true once you scrape at full scale — some files
arrive truncated and only fail when something tries to open them properly.

Add a pass here that opens every saved file with PIL inside a `try`/`except`, logs the failures,
deletes the unreadable ones, and reports the count per class. Then extend the table above with a
`readable` column.

## 5. A first look at the images

Before anything else, look at what you actually downloaded. This is the fastest way to discover
that a query returned drawings, product collages, or pictures of the *word* rather than the thing.

In [ ]:
from pathlib import Path
from PIL import Image

n_show = 6
labels = list(summary['classes'].keys())

fig, axes = plt.subplots(len(labels), n_show, figsize=(2.2 * n_show, 2.4 * len(labels)))
axes = np.atleast_2d(axes)

for r, label in enumerate(labels):
    files = sorted(Path(summary['save_dir'], label).glob('*.jpg'))[:n_show]
    for c in range(n_show):
        ax = axes[r, c]
        ax.axis('off')
        if c < len(files):
            ax.imshow(Image.open(files[c]))
            if c == 0:
                ax.set_title(label, loc='left', fontsize=11)

plt.tight_layout()
plt.show()

## 6. Turning the records into a manifest

`records` is one row per saved image. The manifest is the permanent record of what you trained on,
and it matters more here than in a normal project: **search results change over time, so re-running
this scraper later will not give you the same dataset.** The manifest is the only reproducible
account of the run.

The prompt requires at minimum `filename`, `label`, `source_url`, `query`, and a boolean
`readable`. The scraper supplies everything but `readable` — add it from the integrity pass you
write in section 4.1.

Notebook 2 extends this file with a `kept` column. Add to it there; do not overwrite it.

In [ ]:
manifest = pd.DataFrame(records)

# TODO: set this from your integrity check in section 4.1, not to a constant.
manifest['readable'] = True

Path('data').mkdir(exist_ok=True)
manifest.to_csv('data/P2_manifest.csv', index=False)

print(f'{len(manifest)} rows written to data/P2_manifest.csv')
manifest.head()

## Review

This notebook built a small labeled image dataset from scratch:

1. **The scraper** (`ddgs_scraper.py`) searches DuckDuckGo, downloads results, filters by size and
   aspect ratio, and saves one folder per class
2. **Classes and queries** are declared as a dictionary — several queries per class, because one
   query returns one narrow slice of a category
3. **Running it** produced both the images on disk and a record per image
4. **Accounting** tracked requested → saved per class
5. **Looking at the images** is the fastest way to catch a query that returned the wrong thing
6. **The manifest** is the reproducible record of a scrape that cannot be reproduced by re-running

### To turn this into your P2 Notebook 1

- [ ] Replace `CLASS_QUERIES` with your own 3–10 classes, and justify each in a markdown cell:
      why these, what makes them separable, where you expect the model to struggle
- [ ] State, before you model anything, which **pair of classes you expect to be confused most**.
      Notebook 3 will check you
- [ ] Justify every argument you pass to `build_image_dataset`
- [ ] Scrape at full scale — aim to save 300–400 per class to land 200 after cleaning
- [ ] Add the download-integrity pass described in section 4.1
- [ ] Complete the requested → downloaded → readable accounting table
- [ ] Flag any class at risk of falling under 200 and **re-scrape it now**, not after Notebook 2
- [ ] Export the manifest with a real `readable` column

**Next:** `P2_Images-2_Preprocessing_<TeamName>.ipynb` turns this folder of search results into
an actual dataset.